# 04 — Sigma estimation across SNR levels

This notebook characterises how the sigma-only fit behaves as the signal-to-noise ratio degrades. It reuses the reproduced loss and Adam scan from notebook 03 and fits the spread of a single known Gaussian under a sweep of additive-noise levels, repeated over several seeds.

Modules exercised (reproduced on CPU):

- `pipelines.param_pipeline.sigma.SigmaScan` loss and Adam scan (reproduced inline)

Fixed master seed, CPU only.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


In [ ]:
def per_pixel_pred(sigmas, height_axis, amps, mus):
    safe_s2 = 2.0 * np.maximum(sigmas, 1e-6) ** 2
    diff    = height_axis[None, :] - mus[:, None]
    expon   = np.clip(-(diff ** 2) / safe_s2[:, None], -100.0, 0.0)
    return (amps[:, None] * np.exp(expon)).sum(axis=0)

def per_pixel_loss(sigmas, height_axis, profile, amps, mus):
    pred = per_pixel_pred(sigmas, height_axis, amps, mus)
    return float(np.mean((pred - profile) ** 2))

def loss_grad(sigmas, height_axis, profile, amps, mus, eps=1e-5):
    g = np.zeros_like(sigmas)
    for k in range(sigmas.size):
        sp = sigmas.copy(); sp[k] += eps
        sm = sigmas.copy(); sm[k] -= eps
        g[k] = (per_pixel_loss(sp, height_axis, profile, amps, mus) - per_pixel_loss(sm, height_axis, profile, amps, mus)) / (2 * eps)
    return g

def adam_scan(sigmas_init, height_axis, profile, amps, mus, sigma_lower, sigma_upper, n_steps=400, lr=0.2, b1=0.95, b2=0.999):
    eps = 1e-8
    s   = np.clip(sigmas_init.astype(np.float64), sigma_lower, sigma_upper)
    m   = np.zeros_like(s)
    v   = np.zeros_like(s)
    for t in range(n_steps):
        g  = loss_grad(s, height_axis, profile, amps, mus)
        m  = b1 * m + (1.0 - b1) * g
        v  = b2 * v + (1.0 - b2) * g * g
        tf = t + 1.0
        s  = s - lr * (m / (1.0 - b1 ** tf)) / (np.sqrt(v / (1.0 - b2 ** tf)) + eps)
        s  = np.clip(s, sigma_lower, sigma_upper)
    return s

print('helpers ready')

## SNR sweep

For each noise standard deviation we draw several noisy realisations of the same true Gaussian, fit the spread, and record the recovered sigma. SNR is reported as peak amplitude over noise standard deviation.

In [ ]:
H           = 200
height_axis = np.linspace(0.0, 40.0, H).astype(np.float64)
dh          = float(height_axis[1] - height_axis[0])
h_span      = float(height_axis[-1] - height_axis[0])
sigma_lower = dh
sigma_upper = h_span / 2.0

true_amp, true_mu, true_sigma = 1.0, 20.0, 3.0
amps = np.array([true_amp])
mus  = np.array([true_mu])

noise_levels = [0.005, 0.02, 0.05, 0.1, 0.2, 0.4]
n_trials     = 12

results = {}
for nl in noise_levels:
    fits = []
    for _ in range(n_trials):
        prof = gaussian_mixture_profile(height_axis, [(true_amp, true_mu, true_sigma)], noise_std=nl, rng=rng).astype(np.float64)
        s0   = np.array([0.6 * true_sigma])
        sf   = adam_scan(s0, height_axis, prof, amps, mus, sigma_lower, sigma_upper)
        fits.append(float(sf[0]))
    results[nl] = np.array(fits)

for nl in noise_levels:
    snr = true_amp / nl
    print(f'noise={nl:.3f}  SNR={snr:6.1f}  fitted sigma mean={results[nl].mean():.3f}  std={results[nl].std():.3f}')

In [ ]:
snrs    = [true_amp / nl for nl in noise_levels]
means   = np.array([results[nl].mean() for nl in noise_levels])
stds    = np.array([results[nl].std()  for nl in noise_levels])

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.axhline(true_sigma, color='tab:green', lw=1.5, ls='-', label='true sigma')
ax.errorbar(snrs, means, yerr=stds, fmt='o-', color='tab:red', capsize=3, label='fitted sigma (mean +/- std)')
ax.set_xscale('log')
ax.set_xlabel('SNR (peak amplitude / noise std)')
ax.set_ylabel('sigma [m]')
ax.set_title('Sigma estimation vs SNR')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## Expected outcome

At high SNR the fitted sigma should sit on the green true-sigma line with a small spread across trials. As SNR drops the mean estimate should remain near the truth while the trial-to-trial variance grows, illustrating that the spread estimate is unbiased but increasingly uncertain under noise.